# 05 — Cost Analysis

Compare training time, tuning cost, and the performance-vs-compute trade-off.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

test_df = pd.read_csv('../results/aggregated/test_results.csv')
fold_df = pd.read_csv('../results/aggregated/fold_results.csv')

# Early assertion: fail clearly if timing data is missing, rather than silently producing empty plots.
assert 'train_time_s' in fold_df.columns, (
    "Column 'train_time_s' is missing from fold_results.csv. "
    "Ensure training scripts record and save timing information before running this notebook."
)

In [ ]:
# Training time comparison
if 'train_time_s' in fold_df.columns:
    time_summary = fold_df.groupby('model')['train_time_s'].agg(['mean', 'std', 'sum'])
    time_summary = time_summary.sort_values('mean')

    fig, ax = plt.subplots(figsize=(10, 6))
    time_summary['mean'].plot.barh(ax=ax, xerr=time_summary['std'],
                                    color='steelblue', capsize=3)
    ax.set_xlabel('Mean Training Time per Fold (seconds)')
    ax.set_title('Average Training Time per Model')
    plt.tight_layout()
    plt.savefig('../results/figures/training_time.png', dpi=300, bbox_inches='tight')
    plt.savefig('../results/figures/training_time.pdf', bbox_inches='tight')
    plt.show()

    print("\nTraining time summary (seconds):")
    display(time_summary)

In [ ]:
# Tuning cost (total time including Optuna trials)
if 'tuning_time_s' in test_df.columns:
    tuning_summary = test_df.groupby('model')['tuning_time_s'].agg(['mean', 'sum'])
    tuning_summary = tuning_summary.sort_values('mean')

    fig, ax = plt.subplots(figsize=(10, 6))
    tuning_summary['mean'].plot.barh(ax=ax, color='coral')
    ax.set_xlabel('Mean Tuning Time per Dataset (seconds)')
    ax.set_title('Hyperparameter Tuning Cost')
    plt.tight_layout()
    plt.savefig('../results/figures/tuning_cost.png', dpi=300, bbox_inches='tight')
    plt.savefig('../results/figures/tuning_cost.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# Performance vs training time scatter with true Pareto front
# A model is Pareto-optimal if no other model is both faster AND achieves higher performance.
# Pareto-optimal models are highlighted with a star marker and connected by a step-line.

model_families = {
    'xgboost': 'GBDT', 'lightgbm': 'GBDT', 'catboost': 'GBDT',
    'ft_transformer': 'DL', 'tabnet': 'DL', 'saint': 'DL',
    'tabm': 'DL', 'mlp': 'DL', 'realmlp': 'DL',
    'tabpfn': 'FM'
}

# Colour each point by model family
FAMILY_COLORS = {'GBDT': '#2196F3', 'DL': '#FF9800', 'FM': '#4CAF50'}

merged = test_df.copy()
merged['family'] = merged['model'].map(model_families)


def compute_pareto_front(df, time_col, perf_col, higher_is_better=True):
    """
    Return a boolean mask of Pareto-optimal rows.
    A point is Pareto-optimal if no other point has BOTH lower time AND better performance.
    """
    times = df[time_col].values
    perfs = df[perf_col].values
    n = len(df)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            # j dominates i if j is faster and at least as good (or strictly better)
            faster = times[j] < times[i]
            if higher_is_better:
                better_or_equal = perfs[j] >= perfs[i]
                strictly_better = perfs[j] > perfs[i]
            else:
                better_or_equal = perfs[j] <= perfs[i]
                strictly_better = perfs[j] < perfs[i]
            if faster and better_or_equal and (faster or strictly_better):
                is_pareto[i] = False
                break
    return is_pareto


for task_type, metric, higher_is_better in [
    ('binary', 'roc_auc', True),
    ('multiclass', 'accuracy', True),
    ('regression', 'r2', True),
]:
    subset = merged[
        (merged['task_type'] == task_type) &
        (metric in merged.columns) &
        ('train_time_s' in merged.columns)
    ].copy()

    if subset.empty or metric not in subset.columns or 'train_time_s' not in subset.columns:
        continue

    # Aggregate to one (time, performance) point per model by taking mean across datasets
    model_summary = subset.groupby('model').agg(
        train_time_s=('train_time_s', 'mean'),
        performance=(metric, 'mean'),
        family=('family', 'first'),
    ).reset_index()

    pareto_mask = compute_pareto_front(model_summary, 'train_time_s', 'performance', higher_is_better)
    model_summary['pareto'] = pareto_mask

    fig, ax = plt.subplots(figsize=(11, 7))

    # Plot all models, coloured by family
    for _, row in model_summary.iterrows():
        color = FAMILY_COLORS.get(row['family'], 'grey')
        marker = '*' if row['pareto'] else 'o'
        size = 250 if row['pareto'] else 80
        ax.scatter(row['train_time_s'], row['performance'],
                   color=color, marker=marker, s=size, alpha=0.9,
                   label=f"{row['model']} ({row['family']})")
        ax.annotate(row['model'], (row['train_time_s'], row['performance']),
                    textcoords='offset points', xytext=(6, 4), fontsize=8)

    # Draw the Pareto front as a step line connecting Pareto-optimal points
    pareto_pts = model_summary[model_summary['pareto']].sort_values('train_time_s')
    if len(pareto_pts) >= 2:
        ax.step(pareto_pts['train_time_s'], pareto_pts['performance'],
                where='post', linestyle='--', color='red', linewidth=1.5,
                label='Pareto front', zorder=1)

    ax.set_xscale('log')
    ax.set_xlabel('Mean Training Time (seconds, log scale)')
    ax.set_ylabel(f'Mean {metric}')
    ax.set_title(f'{task_type.title()}: Performance vs Training Cost\n(★ = Pareto-optimal)')

    # Deduplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

    plt.tight_layout()
    plt.savefig(f'../results/figures/pareto_{task_type}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'../results/figures/pareto_{task_type}.pdf', bbox_inches='tight')
    plt.show()

    pareto_models = model_summary[model_summary['pareto']]['model'].tolist()
    print(f"\nPareto-optimal models for {task_type}: {pareto_models}")

In [ ]:
# Total compute budget table
if 'train_time_s' in test_df.columns and 'tuning_time_s' in test_df.columns:
    total_time = test_df.groupby('model').agg({
        'train_time_s': 'sum',
        'tuning_time_s': 'sum',
    }).rename(columns={
        'train_time_s': 'total_train_s',
        'tuning_time_s': 'total_tune_s',
    })
    total_time['total_s'] = total_time['total_train_s'] + total_time['total_tune_s']
    total_time['total_hours'] = total_time['total_s'] / 3600
    total_time = total_time.sort_values('total_s')

    print("\nTotal Compute Budget:")
    display(total_time)
else:
    print("Timing columns (train_time_s / tuning_time_s) not found in test_df.")

In [ ]:
# Inference time analysis
# Looks for 'inference_time_s' column in test_df or fold_df.
# If missing, prints a code template showing how to measure inference time.

if 'inference_time_s' in test_df.columns:
    inf_summary = test_df.groupby('model')['inference_time_s'].agg(['mean', 'std', 'min', 'max'])
    inf_summary = inf_summary.sort_values('mean')

    fig, ax = plt.subplots(figsize=(10, 6))
    inf_summary['mean'].plot.barh(ax=ax, xerr=inf_summary['std'],
                                   color='mediumseagreen', capsize=3)
    ax.set_xlabel('Mean Inference Time (seconds)')
    ax.set_title('Model Inference Time Comparison')
    plt.tight_layout()
    plt.savefig('../results/figures/inference_time.png', dpi=300, bbox_inches='tight')
    plt.savefig('../results/figures/inference_time.pdf', bbox_inches='tight')
    plt.show()

    print("\nInference time summary (seconds):")
    display(inf_summary)

elif 'inference_time_s' in fold_df.columns:
    inf_summary = fold_df.groupby('model')['inference_time_s'].agg(['mean', 'std'])
    inf_summary = inf_summary.sort_values('mean')

    fig, ax = plt.subplots(figsize=(10, 6))
    inf_summary['mean'].plot.barh(ax=ax, xerr=inf_summary['std'],
                                   color='mediumseagreen', capsize=3)
    ax.set_xlabel('Mean Inference Time per Fold (seconds)')
    ax.set_title('Model Inference Time Comparison (per fold)')
    plt.tight_layout()
    plt.savefig('../results/figures/inference_time.png', dpi=300, bbox_inches='tight')
    plt.savefig('../results/figures/inference_time.pdf', bbox_inches='tight')
    plt.show()
else:
    print("'inference_time_s' column not found. This is a TEMPLATE for benchmarking inference time.\n")
    print("""
TEMPLATE — measure inference time in your evaluation loop:

    import time

    t0 = time.perf_counter()
    predictions = model.predict(X_test)
    inference_time_s = time.perf_counter() - t0

    results['inference_time_s'] = inference_time_s
    # Then save results as usual to fold_results.csv / test_results.csv

Once inference_time_s is in the results, re-run this cell to see the plot.
""")

## Inference Time Analysis

Inference (prediction) time is often more important than training time in deployment scenarios.
The analysis below looks for `inference_time_s` in the results files.
If not present, a template is provided showing how to benchmark it.